Structured Output 

Models can be requested to provide their response in an format matching in a given schema. This is useful for ensuring the output can be easily parsed and used in a subsquent processing. Langchain supports multiple schema types and methods for enforcing structured ouptut

Pydantic

Pydantic models provide the richest feature set with field validation,description, and nested structures

In [6]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

In [7]:

from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str=Field(description="the title of the movie")
    year:int=Field(description="the year of the movie release")
    director:str=Field(description="the director of the movie")
    rating:float=Field(descrption="the rating of the movie")

C:\Users\huzef\AppData\Local\Temp\ipykernel_39056\2154626271.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'descrption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  rating:float=Field(descrption="the rating of the movie")


In [8]:
model_with_structured_output=model.with_structured_output(Movie)


In [9]:
model_with_structured_output.invoke("name of the movie is Dhamal")

Movie(title='Dhamal', year=2007, director='Priyadarshan', rating=6.5)

Message output alongside parsed structure

In [ ]:
model_with_structured_output=model.with_structured_output(Movie,include_raw=True)# gives raw messge also 


Nested strucutre

In [ ]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str

class Movie(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    generes:list[str]
    budget:float | None =Field("Cant find Budget", description="Budget in Crores")


model_with_cast=model.with_structured_output(Movie)


In [15]:
model_with_cast.invoke("Tell me about lagaan")

Movie(title='Lagaan', year=2001, cast=[Actor(name='Aamir Khan', role='Bhuvan Singh'), Actor(name='Gracy Singh', role='Raju')], generes=['Drama', 'Sports'], budget=None)

TypeDict

TypeDict provides a simpler alternative using Python's built in typing, ideal when you dont need runtime validation

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title: Annotated[str,"the title of the movie"]
    year: Annotated[int,'The year of the movie release']
    director:Annotated[str,"The director of the movie"]
    rating:Annotated[float,"the rating of the movies"]

In [22]:

model_typeDict=model.with_structured_output(MovieDict)
response=model_typeDict.invoke("Lagaan")

In [23]:
response

{'director': 'Ashutosh Gowariker',
 'rating': 8.1,
 'title': 'Lagaan',
 'year': 2001}

In [24]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

DataClasses

A data class is a class typically conataining mainly data , although there arent really any restrictions. you create it using the @dataclass decorator

In [25]:
from pydantic import BaseModel , Field
from langchain.agents import create_agent

class Contact(BaseModel):
    Name:str=Field(description="The name of the person")
    email:str=Field(description="the email of the perosn")
    number:int=Field(description="the number of the person")


agent=create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=Contact
)

In [29]:
from langchain_core.messages import content
result=agent.invoke({
    "messages":[{"role":"user",
    "content":"extract the details of the person from the data RAhul sharma sharma@@gmail.com and 7894561232"}]
})

print(result["structured_response"])

Name='Rahul sharma' email='sharma@@gmail.com' number=7894561232


In [ ]:
import email
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact details of the person"""
    name:str
    email:str
    phones:str

model_dataclass=model.with_structured_output(ContactInfo)

     

In [31]:
respons_of_dataclass=model_dataclass.invoke("extract the details of the person from the data RAhul sharma sharma@@gmail.com and 7894561232")
respons_of_dataclass

{'email': 'sharma@@gmail.com', 'name': 'RAhul sharma', 'phones': '7894561232'}